In [ ]:
# =============================================================
#  Plaka Tanima — YOLO11m + BoT-SORT + fast-alpr
# =============================================================
#  Videodaki araclarin plakalarini okuyup ekrana basan bir pipeline.
#  Avrupa plakalarina gore ayarli; tek bir ulkeye sabitlemedim, format
#  kontrolunu gevsek tuttum (bkz. is_valid_eu_plate).
#
#  Calisma mantigi:
#    1) YOLO11m ile arac tespiti (sadece car / bus / truck)
#    2) BoT-SORT ile takip -> her araca frame'ler boyunca sabit track id
#    3) Arac ekranin alt bandina (zon) girince plaka okumaya basla
#    4) Her track icin birkac gecerli okuma biriktir, en guvenilirini
#       KILITLE ve arac kadrajda kaldigi surece hep onu goster
#
#  Neden kilitleme? fast-alpr ayni plakayi kare kare farkli okuyabiliyor
#  (isik, aci, motion blur yuzunden). Tek frame'e guvenmek yerine birkac
#  okumanin en yuksek confidence'lisini secince sonuc cok daha stabil
#  oluyor; arac zondan ciksa bile plaka ekranda sabit kaliyor.
#
#  Kurulum:
#    pip install ultralytics opencv-python
#    pip install "fast-alpr[onnx-gpu]"   # CUDA'li NVIDIA GPU icin
# =============================================================

In [ ]:
# Kütüphaneleri yüklüyorum
import re
from collections import defaultdict

import cv2
import numpy as np

import onnxruntime as ort
from ultralytics import YOLO
from fast_alpr import ALPR

In [ ]:
# ---- Ayarlar ----
VIDEO_PATH  = "video.mp4"           # girdi videosu
OUTPUT_PATH = "output_plaka.mp4"    # cikti videosu
DEVICE      = 0                      # YOLO icin GPU id

VEHICLE_CLASSES = [2, 5, 7]          # COCO: car, bus, truck
VEHICLE_CONF    = 0.40               # arac tespit esigi

ZONE_TOP_RATIO    = 0.70             # 7/10  -> zon ust siniri
ZONE_BOTTOM_RATIO = 0.90             # 9/10  -> zon alt siniri
N_VALID_READS     = 5                # kilitlemeden once gereken gecerli okuma
PLATE_MIN_CONF    = 0.50             # bir okumayi adaya almak icin OCR esigi

SHOW_WINDOW = False                  # True -> cv2.imshow ile canli izle

# ---- Cizim olcekleri (4K / 3840x2160 icin buyutuldu) ----
BOX_THICK   = 5                      # arac & plaka kutu kalinligi
ZONE_THICK  = 3                      # zon cizgi kalinligi
FONT_SCALE  = 2.2                    # metin boyutu
FONT_THICK  = 5                      # metin kalinligi

# ---- Renkler (BGR) ----
COLOR_PLATE = (211, 0, 148)          # plaka kutusu & yazi zemini (mor)
COLOR_TEXT  = (255, 255, 255)        # plaka yazisi (beyaz)
COLOR_ZONE  = (255, 0, 0)            # zon cizgileri (mavi)

In [ ]:
# ---- Modeller ----
vehicle_model = YOLO("yolo11m.pt")

# ORT'nin GPU EP'yi gorebilmesi icin CUDA/cuDNN DLL'lerini (torch icinden)
# onceden yuklemek lazim. Bunu ALPR()'dan ONCE cagirmak sart, cunku ilk
# InferenceSession orada aciliyor; gec kalirsan CPU'ya dusebiliyor.
try:
    ort.preload_dlls()
except Exception as e:
    print("preload_dlls atlandi:", e)

alpr = ALPR(
    detector_model="yolo-v9-t-384-license-plate-end2end",
    ocr_model="cct-s-v2-global-model",
)


In [ ]:
# ---- Plaka dogrulama ----
# EU plakalari ulkeden ulkeye cok degistigi icin kati bir ulke-regex'i
# yerine basit bir format kontrolu yapiyorum: 5-9 alfanumerik, icinde en
# az 1 harf + 1 rakam. Tek bir ulkeye sabitlemek istersen burayi sertlestir.
def normalize_plate(text):
    # fast-plate-ocr'in birakabilecegi padding/bosluk karakterlerini temizle
    return re.sub(r"[^A-Z0-9]", "", (text or "").upper())


def is_valid_eu_plate(text):
    t = normalize_plate(text)
    if not (5 <= len(t) <= 9):
        return False
    if not any(c.isdigit() for c in t):
        return False
    if not any(c.isalpha() for c in t):
        return False
    return True

In [ ]:
# ---- Track durumu ----
# track_id -> durum sozlugu. Her arac icin okuma gecmisini ve kilitlenmis
# plakayi burada tutuyorum.
def yeni_durum():
    return {
        "candidates": [],        # [(plate_text, conf), ...]  (kilide kadar)
        "locked_text": None,     # kilitlenmis plaka metni
        "locked_conf": None,     # kilitlenmis plakanin confidence'i
        "last_plate_box": None,  # full-frame plaka kutusu (cizim icin)
        "running_text": None,    # toplama asamasinda anlik okuma
    }


track_state = defaultdict(yeni_durum)

In [ ]:
# ---- Plaka okuma yardimcisi ----
def en_iyi_plaka(car_crop):
    """car_crop uzerinde fast-alpr calistir, en yuksek OCR confidence'li
    GECERLI plakayi dondur.

    Donen: (text, conf, (px1, py1, px2, py2)) | None
    Koordinatlar crop'a goredir; cizim sirasinda araç offset'i (x1, y1)
    eklenerek full-frame'e tasiniyor."""
    if car_crop is None or car_crop.size == 0:
        return None

    results = alpr.predict(car_crop)
    best = None
    for r in results:
        if r.ocr is None:
            continue
        if not is_valid_eu_plate(r.ocr.text):
            continue

        # fast-plate-ocr karakter-bazli confidence listesi donebiliyor ->
        # plaka geneli icin ortalamasini al (tek float gelirse oldugu gibi kullan)
        c = r.ocr.confidence
        conf = (float(np.mean(c))
                if isinstance(c, (list, tuple, np.ndarray)) and len(c)
                else float(c))

        bb = r.detection.bounding_box
        box = (int(bb.x1), int(bb.y1), int(bb.x2), int(bb.y2))

        if best is None or conf > best[1]:
            best = (normalize_plate(r.ocr.text), conf, box)
    return best


In [ ]:
# ---- Cizim yardimcilari ----
def kutu_ciz(frame, box, color, thickness=None):
    if thickness is None:
        thickness = BOX_THICK
    x1, y1, x2, y2 = box
    cv2.rectangle(frame, (x1, y1), (x2, y2), color, thickness)


def etiket_ciz(frame, text, org, bg_color, text_color=(0, 0, 0)):
    x, y = org
    y = max(y, int(36 * FONT_SCALE))  # ust kenardan tasmasin
    (tw, th), bl = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX,
                                   FONT_SCALE, FONT_THICK)
    pad = int(6 * FONT_SCALE)
    cv2.rectangle(frame, (x, y - th - bl - pad), (x + tw + pad, y), bg_color, -1)
    cv2.putText(frame, text, (x + pad // 2, y - bl - pad // 3),
                cv2.FONT_HERSHEY_SIMPLEX, FONT_SCALE, text_color,
                FONT_THICK, cv2.LINE_AA)

In [ ]:
# ---- Ana dongu ----
cap = cv2.VideoCapture(VIDEO_PATH)
fps    = cap.get(cv2.CAP_PROP_FPS) or 25.0
width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

zone_y_top    = int(height * ZONE_TOP_RATIO)
zone_y_bottom = int(height * ZONE_BOTTOM_RATIO)

writer = cv2.VideoWriter(
    OUTPUT_PATH, cv2.VideoWriter_fourcc(*"mp4v"), fps, (width, height)
)

frame_idx = 0
while True:
    ok, frame = cap.read()
    if not ok:
        break
    frame_idx += 1

    # OCR crop'larini temiz goruntuden alabilmek icin cizimden once kopya al
    clean = frame.copy()

    # BoT-SORT ile takip (persist=True -> id'ler frame'ler arasi korunur)
    results = vehicle_model.track(
        frame,
        persist=True,
        tracker="botsort.yaml",
        classes=VEHICLE_CLASSES,
        conf=VEHICLE_CONF,
        device=DEVICE,
        verbose=False,
    )[0]

    if results.boxes is not None and results.boxes.id is not None:
        boxes = results.boxes.xyxy.cpu().numpy()
        ids   = results.boxes.id.cpu().numpy().astype(int)

        for (bx1, by1, bx2, by2), tid in zip(boxes, ids):
            x1, y1, x2, y2 = int(bx1), int(by1), int(bx2), int(by2)
            cx = (x1 + x2) // 2
            cy = (y1 + y2) // 2

            st = track_state[tid]
            in_zone = zone_y_top <= cy <= zone_y_bottom

            # --- zon icindeyse plaka oku ---
            if in_zone:
                cx1, cy1 = max(0, x1), max(0, y1)
                cx2, cy2 = max(0, x2), max(0, y2)
                crop = clean[cy1:cy2, cx1:cx2]
                best = en_iyi_plaka(crop)
                if best is not None:
                    ptext, pconf, (px1, py1, px2, py2) = best
                    # crop koordinatini full-frame'e tasi
                    st["last_plate_box"] = (cx1 + px1, cy1 + py1,
                                            cx1 + px2, cy1 + py2)
                    if st["locked_text"] is None:
                        st["running_text"] = ptext
                        if pconf >= PLATE_MIN_CONF:
                            st["candidates"].append((ptext, pconf))
                            if len(st["candidates"]) >= N_VALID_READS:
                                # yeterli okuma biriktiyse en guvenilirini kilitle
                                best_c = max(st["candidates"],
                                             key=lambda c: c[1])
                                st["locked_text"], st["locked_conf"] = best_c

            # --- cizim (araca rectangle YOK) ---
            # plaka kutusu: sadece zon icinde taze tespit varken (mor)
            if in_zone and st["last_plate_box"] is not None:
                kutu_ciz(frame, st["last_plate_box"], COLOR_PLATE)

            # plaka yazisi: sadece KILITLENMIS plaka (mor zemin + beyaz yazi).
            # arac zondan ciksa da takip ettigi surece gosterilir.
            if st["locked_text"] is not None:
                etiket_ciz(frame, st["locked_text"], (x1, y1),
                           COLOR_PLATE, COLOR_TEXT)

    # zon bandini gorsel referans olarak ciz (OCR'dan SONRA, crop'a karismasin)
    cv2.line(frame, (0, zone_y_top), (width, zone_y_top), COLOR_ZONE, ZONE_THICK)
    cv2.line(frame, (0, zone_y_bottom), (width, zone_y_bottom), COLOR_ZONE, ZONE_THICK)

    writer.write(frame)
    if SHOW_WINDOW:
        cv2.imshow("ALPR", frame)
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break

cap.release()
writer.release()
if SHOW_WINDOW:
    cv2.destroyAllWindows()
print(f"Bitti -> {OUTPUT_PATH}")